# 03 — Strength progression EDA
Completed-session, non-warmup weighted-set summaries using the product's maximum-load representative rule. Plateau-like patterns below are descriptive heuristics, not predictions.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from metaform_ds.features import add_strength_features
from metaform_ds.utils.notebooks import load_notebook_data
data = load_notebook_data(); strength = add_strength_features(data.strength)
print(f'DATA MODE: {data.mode}; rows: {len(strength)}')
if data.mode == 'SYNTHETIC': print('SYNTHETIC demonstration only.')

In [ ]:
counts = strength.groupby(['source_type', 'source_key_anonymous'], dropna=False).size().sort_values(ascending=False)
if counts.empty: print('Insufficient data: no qualifying strength sessions.')
else:
    fig, ax = plt.subplots(); counts.head(15).plot.bar(ax=ax); ax.set(title='Qualifying sessions per strength source', xlabel='Source', ylabel='Completed sessions'); plt.show()

In [ ]:
if not strength.empty:
    source = strength['source_key_anonymous'].iloc[0]; view = strength[strength['source_key_anonymous'].eq(source)]
    fig, ax = plt.subplots(); ax.plot(view['session_date'], view['max_weight_kg'], marker='o'); ax.set(title='Representative maximum load progression', xlabel='Session date (UTC)', ylabel='Maximum load (kg)'); fig.autofmt_xdate(); plt.show()

In [ ]:
if not strength.empty:
    fig, ax = plt.subplots(); ax.plot(view['session_date'], view['total_volume_kg'], marker='o'); ax.set(title='Training volume progression', xlabel='Session date (UTC)', ylabel='Volume (kg × reps)'); fig.autofmt_xdate(); plt.show()

In [ ]:
if strength.empty: print('No history for PR-frequency analysis.')
else:
    running_prior = strength.groupby(['anonymous_user_id', 'source_key_anonymous'])['max_weight_kg'].transform(lambda s: s.shift(1).cummax())
    pr = running_prior.isna() | strength['max_weight_kg'].gt(running_prior)
    monthly_prs = strength.assign(is_pr=pr).set_index('session_date').resample('MS')['is_pr'].sum()
    fig, ax = plt.subplots(); ax.bar(monthly_prs.index, monthly_prs.values, width=20); ax.set(title='Descriptive PR frequency', xlabel='Month (UTC)', ylabel='New running maxima'); fig.autofmt_xdate(); plt.show()

In [ ]:
gaps = strength['days_since_previous_session'].dropna()
if gaps.empty: print('Insufficient data for time-between-session analysis.')
else:
    fig, ax = plt.subplots(); ax.hist(gaps, bins=min(12, max(3, len(gaps)))); ax.set(title='Time between source sessions', xlabel='Days', ylabel='Intervals'); plt.show()
insufficient = counts[counts < 3] if not counts.empty else counts; print(f'Sources with fewer than 3 sessions: {len(insufficient)}')
if strength.empty: print('No plateau-like pattern review is possible.')
else:
    recent = strength.groupby(['anonymous_user_id', 'source_key_anonymous']).tail(4).groupby(['anonymous_user_id', 'source_key_anonymous'])['max_weight_kg'].agg(['count', 'min', 'max'])
    plateau_like = recent[(recent['count'] >= 4) & ((recent['max'] - recent['min']) <= 1.0)]
    print(f'Descriptive plateau-like recent ranges (≤1 kg across 4 observations): {len(plateau_like)}. This is not an ML prediction or training-quality judgment.')